In [1]:
import time

import numpy as np

import timm
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader

# Evaluate Model

In [2]:
def evaluate_model(model, test_loader, device, criterion):
    since = time.time()
    model.eval()
    running_loss = 0.0
    running_corrects = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            running_corrects += (labels == predicted).sum().item()

    epoch_loss = running_loss / test_dataset_size
    epoch_accuracy = 100 * running_corrects / test_dataset_size

    print(f"Eval Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%")
    print(f'Got {running_corrects} out of {test_dataset_size} images correctly')
    
    time_elapsed = time.time() - since
    print(f'Evaluation complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    return epoch_loss, epoch_accuracy

# Practice Test Dataset from Kaggle

In [3]:
test_dataset_path = './prac_test/'

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_dataset = torchvision.datasets.ImageFolder(
    root=test_dataset_path, 
    transform=test_transform
)

test_loader = torch.utils.data.DataLoader(  
    dataset=test_dataset, 
    batch_size=32, 
    shuffle=True
)

num_classes = 4 # healthy, cordona, pestalotiopsis, sigatoka
swin_model = timm.create_model('swin_tiny_patch4_window7_224', pretrained=False, num_classes = 4, global_pool='avg')
swin_model_trained = "VIT_transfer_learning/models/swin_tiny_1.pth"
state_dict = torch.load(swin_model_trained)
swin_model.load_state_dict(state_dict)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")
test_dataset_size = len(test_dataset)
loss_function = nn.CrossEntropyLoss()
loss, accuracy = evaluate_model(swin_model, test_loader, device, loss_function)


Using cpu device


C:\Users\phea\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))
C:\Users\phea\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\PIL\JpegImagePlugin.py:890: UserWarning: Image appears to be a malformed MPO file, it will be interpreted as a base JPEG file
  warnings.warn(


Eval Loss: 0.0300, Accuracy: 99.33%
Got 2520 out of 2537 images correctly
Evaluation complete in 2m 25s
